In [ ]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from time import sleep
import os
import requests
from bs4 import BeautifulSoup
import re


In [2]:

# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'CY CBTRNC'
print(f"Running {regulatorName} Web Scraping Tool v.1.1")

now=datetime.datetime.now()

filename= f'{regulatorName} SQL Ready {str(now).replace(":",".")[:-7]}.xlsx'

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)





Running CY CBTRNC Web Scraping Tool v.1.1


In [3]:

# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------


def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict




In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = {
    regulatorName+' 1' : 'https://mb.gov.ct.tr/en/bilgiler/bankalar',
    regulatorName+' 2' : 'https://mb.gov.ct.tr/en/node/4029',
    }

Typology = {

            regulatorName+" 1": "List of Banks",
            regulatorName+" 2": "International Banking Units",

            }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

          'Phone - Mother company': [], 'Check': []}





processdate = now.strftime('%Y-%m-%d')



In [5]:

# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

#driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=chromeOptions )

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



In [ ]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for index, reg in enumerate(regdict):

    print(f"[INFO] : Working {index+1}/{len(regdict)} _({reg})_ ")

    sleep(3)
    # Fetch the page content
    response = requests.get(regdict[reg], verify=False)
    response.encoding = 'utf-8'  # Ensure correct encoding
    soup = BeautifulSoup(response.text, 'html.parser')
    if index == 0:
        tables = soup.find_all('table')

        # Step 3: Process each table
        for table_index, table in enumerate(tables):
        
            # Extract headers
            headers = [th.get_text(strip=True) for th in table.find_all('th')]

            # Extract rows
            rows = table.find_all('tr')[1:]  # Skip header row
            for row_index, row in enumerate(rows):
                cells = row.find_all(['td', 'th'])
                values = [cell.get_text(strip=True) for cell in cells]
                # Assign each column to a variable
                for col_index, header in enumerate(headers):
                    var_name = header.replace(" ", "_").lower()
                    var_value = values[col_index] if col_index < len(values) else ''
                    #print(f"{var_name} = '{var_value}'")
                    if var_name == 'bank_code':
                        sqldict['InternalID_1_type'].append('Bank Code')
                        sqldict['InternalID_1'].append(var_value)
                    elif var_name == 'bank_name':
                        sqldict['Name'].append(var_value)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['RegCtry'].append(reg.split()[0])
                        sqldict['RegCode'].append(reg.split()[1])
                        sqldict['ListCode'].append(reg.split()[2])
                        sqldict['ListName'].append(Typology[reg])
                        sqldict['RegulationType'].append('Regulated')
                    elif var_name == 'type':
                        sqldict['Typology'].append(var_value)
                    elif var_name == 'tel._no.':
                        sqldict['Phone'].append(var_value)
                    elif var_name == 'fax_no.':
                        sqldict['Fax'].append(var_value)
                    elif var_name == 'website':
                        sqldict['Website'].append(var_value)
                    elif var_name == 'address':
                        sqldict['Address_1'].append(var_value)
                sqldict = bourange_same_length_array(sqldict)
    elif index ==1:

        heading_text = "International Banking Units Continuing Their Commercial Activities"

        # 1) Locate the heading paragraph
        heading_p = soup.find("p", string=lambda t: t and heading_text in t)
        if not heading_p:
            raise ValueError("Heading not found")

        banks = []
        for p in heading_p.find_next_siblings("p"):
            # stop when next bold/strong section starts
            if p.find("strong"):
                break
            text = p.get_text(" ", strip=True).replace("\xa0", " ")
            m = re.match(r"\d+\)\s*(.+)", text)
            if m:
                banks.append(m.group(1))

        for bank in banks:
            sqldict['Name'].append(bank)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegulationType'].append('Regulated')
        sqldict = bourange_same_length_array(sqldict)
                

[INFO] : Working 1/2 _(CY CBTRNC 1)_ 
[INFO] : Working 2/2 _(CY CBTRNC 2)_ 


In [ ]:

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)
df.to_excel(filename,index=False)